In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error, r2_score

# %% 1. Configuration & Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("../outputs")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

# Ensure this path matches your saved file name
NPZ_FILE = OUTPUT_DIR / "dl_sequences.npz" 

# %% 2. The Corrected Lazy Dataset Class
class ICULazyDataset(Dataset):
    def __init__(self, npz_path, split='train'):
        self.npz_path = npz_path
        self.split = split
        self.seq_key = f'X_{split}_seq'
        self.static_key = f'X_{split}_static'
        self.label_key = f'y_{split}_cont'
        
        # We only open the file briefly to get the length
        temp_data = np.load(self.npz_path, mmap_mode='r')
        self.length = temp_data[self.label_key].shape[0]
        
        # We DO NOT store the file handle here (self.data = None)
        # This allows the object to be pickled and sent to worker processes
        self.handle = None 

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Open the file handle only when the worker process needs it
        if self.handle is None:
            self.handle = np.load(self.npz_path, mmap_mode='r')
            
        sequence = torch.from_numpy(self.handle[self.seq_key][idx]).float()
        static = torch.from_numpy(self.handle[self.static_key][idx]).float()
        label = torch.tensor(self.handle[self.label_key][idx]).float()
        
        return {
            'sequence': sequence, 
            'static': static,     
            'label': label       
        }

# %% 3. Initialize DataLoaders
train_ds = ICULazyDataset(NPZ_FILE, split='train')
val_ds   = ICULazyDataset(NPZ_FILE, split='val')
test_ds  = ICULazyDataset(NPZ_FILE, split='test') # Added for Section 7

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=64, num_workers=0)

print(f"Lazy Loaders ready. Training on {len(train_ds)} samples.")

# %% 4. RNN Architecture
class ICURNNRegressor(nn.Module):
    def __init__(self, n_temporal, n_static, hidden_dim=64, n_layers=2, rnn_type='LSTM', dropout=0.3):
        super().__init__()
        
        if rnn_type == 'LSTM':
            self.rnn = nn.LSTM(n_temporal, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        else:
            self.rnn = nn.GRU(n_temporal, hidden_dim, n_layers, batch_first=True, dropout=dropout)
            
        self.static_net = nn.Sequential(
            nn.Linear(n_static, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim + (hidden_dim // 2), hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_seq, x_static):
        _, h_n = self.rnn(x_seq)
        if isinstance(h_n, tuple): h_n = h_n[0] 
        last_h = h_n[-1] 
        static_h = self.static_net(x_static)
        combined = torch.cat([last_h, static_h], dim=1)
        return self.regressor(combined).squeeze(-1)

# %% 5. Early Stopping & Training logic
class EarlyStopping:
    def __init__(self, patience=3, path='checkpoint.pt'):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            torch.save(model.state_dict(), self.path)
        elif val_loss > self.best_loss:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_loss = val_loss
            torch.save(model.state_dict(), self.path)
            self.counter = 0

def train_model(name, model, train_loader, val_loader):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.L1Loss()
    stopper = EarlyStopping(path=MODEL_DIR / f'best_{name.lower()}.pt')
    
    for epoch in range(20):
        model.train()
        train_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            out = model(batch['sequence'].to(DEVICE), batch['static'].to(DEVICE))
            loss = criterion(out, batch['label'].to(DEVICE))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_mae = 0
        with torch.no_grad():
            for batch in val_loader:
                out = model(batch['sequence'].to(DEVICE), batch['static'].to(DEVICE))
                val_mae += torch.abs(torch.clamp(out, min=0) - batch['label'].to(DEVICE)).mean().item()
        
        avg_train = train_loss / len(train_loader)
        avg_val = val_mae / len(val_loader)
        print(f"[{name}] Epoch {epoch+1}: Train Loss {avg_train:.4f}, Val MAE {avg_val:.4f}")
        
        stopper(avg_val, model)
        if stopper.early_stop: 
            print("Early stopping triggered.")
            break
    
    model.load_state_dict(torch.load(MODEL_DIR / f'best_{name.lower()}.pt'))

# %% 6. Execution
# Determine input dimensions from the file
temp_data = np.load(NPZ_FILE, mmap_mode='r')
N_TEMP = temp_data['X_train_seq'].shape[-1]
N_STAT = temp_data['X_train_static'].shape[-1]

all_results = {}
for rnn_type in ['LSTM', 'GRU']:
    print(f"\nStarting training for {rnn_type}...")
    model = ICURNNRegressor(N_TEMP, N_STAT, rnn_type=rnn_type).to(DEVICE)
    train_model(rnn_type, model, train_loader, val_loader)

# %% 7. Final Test Evaluation
print("\n" + "="*30 + "\nFINAL TEST PERFORMANCE\n" + "="*30)
for name in ['LSTM', 'GRU']:
    model = ICURNNRegressor(N_TEMP, N_STAT, rnn_type=name).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_DIR / f'best_{name.lower()}.pt'))
    model.eval()
    
    preds, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            out = model(batch['sequence'].to(DEVICE), batch['static'].to(DEVICE))
            preds.extend(torch.clamp(out, min=0).cpu().numpy())
            actuals.extend(batch['label'].numpy())
            
    print(f"{name} -> MAE: {mean_absolute_error(actuals, preds):.3f} days, R2: {r2_score(actuals, preds):.3f}")
    

Lazy Loaders ready. Training on 20918 samples.

Starting training for LSTM...
[LSTM] Epoch 1: Train Loss 2.0014, Val MAE 1.8861
[LSTM] Epoch 2: Train Loss 1.8655, Val MAE 1.8678
[LSTM] Epoch 3: Train Loss 1.8430, Val MAE 1.8487
[LSTM] Epoch 4: Train Loss 1.8267, Val MAE 1.8416
[LSTM] Epoch 5: Train Loss 1.8150, Val MAE 1.8374
[LSTM] Epoch 6: Train Loss 1.8029, Val MAE 1.8289
[LSTM] Epoch 7: Train Loss 1.7932, Val MAE 1.8299
[LSTM] Epoch 8: Train Loss 1.7777, Val MAE 1.8256
[LSTM] Epoch 9: Train Loss 1.7721, Val MAE 1.8186
[LSTM] Epoch 10: Train Loss 1.7638, Val MAE 1.8188
[LSTM] Epoch 11: Train Loss 1.7531, Val MAE 1.8202
[LSTM] Epoch 12: Train Loss 1.7505, Val MAE 1.8265
Early stopping triggered.

Starting training for GRU...
[GRU] Epoch 1: Train Loss 1.9926, Val MAE 1.8934
[GRU] Epoch 2: Train Loss 1.8684, Val MAE 1.8465
[GRU] Epoch 3: Train Loss 1.8373, Val MAE 1.8352
[GRU] Epoch 4: Train Loss 1.8216, Val MAE 1.8205
[GRU] Epoch 5: Train Loss 1.8014, Val MAE 1.8176
[GRU] Epoch 6: Tra